# Lesson 1 — Your first LLM call

**You will:** send a message to a real Large Language Model and read its reply.

**You will not** use any framework. Just one HTTP-shaped Python call to a free model. Once this lands, every later lesson is just a question of *what wraps around this?*

**Time:** about 5 minutes.

**Reading:** [lesson.md](./lesson.md)

## Step 1 — Install the OpenAI Python client

OpenRouter speaks the OpenAI API. Same client, different endpoint.

In [ ]:
%pip install -q openai

## Step 2 — Get a free API key

1. Open <https://openrouter.ai> in a new tab.
2. Sign in (GitHub or Google).
3. Open your dashboard, create a key, copy it. It starts with `sk-or-`.
4. Paste it when prompted below. The free tier requires no credit card.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (it starts with sk-or-): ")

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")

## Step 3 — Your first LLM call

Three things to notice in the code below:

- **`messages`** is a list of role/content pairs. The model reads them in order.
- The **`system`** role tells the model who it is.
- The **`user`** role is what the user said.

Run the cell and see what comes back.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# Pin a specific free-tier model. If a cell errors with 'model not available',
# swap for any other *:free model from https://openrouter.ai/models.
MODEL = os.environ.get("BAREBEAR_MODEL", "meta-llama/llama-3.2-3b-instruct:free")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful tutor. Keep answers short."},
        {"role": "user", "content": "What is a Large Language Model, in one sentence?"},
    ],
)

print(response.choices[0].message.content)
print("---")
print("model:", response.model)
print("prompt tokens:", response.usage.prompt_tokens)
print("completion tokens:", response.usage.completion_tokens)

## What just happened

- You sent two messages to a real LLM running on someone else's hardware.
- The model produced one reply.
- You can see exactly how many *tokens* went in and how many came out. A token is roughly 3/4 of an English word.

**That is the entire LLM API.** Everything else — chat apps, agents, copilots, search assistants — is built on top of this.

## Step 4 — The system prompt is the agent's personality

Same user message. Different system prompts. Watch the answer change.

In [ ]:
personalities = [
    "You are a helpful tutor. Keep answers short.",
    "You are a Shakespearean poet. Answer only in iambic pentameter.",
    "You only respond in valid JSON. No prose.",
]

user_message = "What is a Large Language Model, in one sentence?"

for persona in personalities:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": user_message},
        ],
    )
    print(f"-- system: {persona[:60]}...")
    print(r.choices[0].message.content)
    print()

## Exercise

Pick **one** and edit the cell below.

1. Change the system prompt so the model behaves like a strict maths teacher walking a student through a quadratic equation.
2. Ask the same question with three different system prompts and compare the answers. Which prompt produced the most useful reply, and why?
3. Print `response.usage.total_tokens`. Estimate the cost in tokens-per-character of the reply.

There is no "right" answer. The point is: notice how much the system prompt changes the output.

In [ ]:
# Your turn — edit and run.

my_system_prompt = ""  # ← write your own system prompt here
my_user_message = ""   # ← and your own question here

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": my_system_prompt},
        {"role": "user", "content": my_user_message},
    ],
)
print(r.choices[0].message.content)

## Preview — the same call, through barebear

Before we move on: the entire reason **barebear** exists is to wrap that chat-completion call inside a *loop* with *tools*. The wrapping doesn't hide the call — it adds policy, budget, and a report on top.

Here's the **exact same model call you just made**, but routed through barebear's `OpenRouterModel` adapter. Notice it returns a `ModelResponse` with the same content and token counts. No magic — just one Python class wrapping the same HTTP call.

If this cell produces output that looks identical to the one above, you've verified the framework is *literally* "that call, plus a loop, plus a report." That's all of Lesson 2 in one preview.

In [ ]:
%pip install -q "barebear[openai]"

from barebear import OpenRouterModel

# OpenRouterModel reads OPENROUTER_API_KEY (and optionally BAREBEAR_MODEL)
# from the environment — the same key you set above.
model = OpenRouterModel()

response = model.complete(
    messages=[
        {"role": "system", "content": "You are a helpful tutor. Keep answers short."},
        {"role": "user", "content": "What is a Large Language Model, in one sentence?"},
    ],
)

print(response.content)
print("---")
print("prompt tokens:", response.prompt_tokens)
print("completion tokens:", response.completion_tokens)

## What's next

In **Lesson 2** we wrap this single call in a loop, give it a tool, and watch a chat completion become an *agent*. That's the moment everything changes.

[Lesson 2 — The agent loop →](../02-the-agent-loop/lesson.md)